# SQLite Webpage Auditor - Hands-on Practice Notebook

This interactive notebook allows you to test, execute, and analyze the **SQLite Webpage Auditor** script directly in Google Colab or your local Jupyter environment.

### Objectives:
1. Setup environment and dependencies.
2. Initialize an SQLite database table.
3. Audit URLs using `requests` and `BeautifulSoup`.
4. Inspect title tags, canonical links, GA4 implementation, and database columns (`gender`).
5. Run SQL reporting queries directly using `pandas`.

In [ ]:
# Step 1: Install required dependencies
!pip install beautifulsoup4 requests openpyxl pandas

In [ ]:
# Step 2: Initialize SQLite Database
import sqlite3
import pandas as pd
import requests
from bs4 import BeautifulSoup

DB_NAME = "webpage_audit.db"

def init_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS audit_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            url TEXT UNIQUE,
            status_code INTEGER,
            title TEXT,
            title_length INTEGER,
            canonical_url TEXT,
            has_ga4 INTEGER,
            gender TEXT
        )
    ''')
    conn.commit()
    conn.close()
    print("Database initialized successfully with updated column schema.")

init_database()

In [ ]:
# Step 3: Define Web Auditor Function
def run_audit(url, sample_gender="unspecified"):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=10)
        status_code = response.status_code
        soup = BeautifulSoup(response.text, 'html.parser')
        
        title = soup.title.string.strip() if (soup.title and soup.title.string) else ""
        title_len = len(title)
        
        canonical = soup.find('link', rel='canonical')
        canonical_url = canonical['href'] if (canonical and 'href' in canonical.attrs) else ""
        
        has_ga4 = 1 if ('googletagmanager.com' in response.text or 'G-' in response.text) else 0
        
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT OR REPLACE INTO audit_log (url, status_code, title, title_length, canonical_url, has_ga4, gender)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (url, status_code, title, title_len, canonical_url, has_ga4, sample_gender))
        conn.commit()
        conn.close()
        print(f"Audited: {url} | Status: {status_code} | Title Length: {title_len}")
    except Exception as e:
        print(f"Error auditing {url}: {e}")

In [ ]:
# Step 4: Run Audit on Sample URLs
sample_urls = [
    "https://www.google.com",
    "https://www.wikipedia.org",
    "https://httpbin.org/status/404"
]

for u in sample_urls:
    run_audit(u)

In [ ]:
# Step 5: Execute SQL Analysis & View DataFrame
conn = sqlite3.connect(DB_NAME)
df = pd.read_sql_query("SELECT id, url, status_code, title, title_length, canonical_url, has_ga4, gender FROM audit_log", conn)
conn.close()

df